In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
torch.set_float32_matmul_precision('high')

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.metrics import (
    roc_curve, f1_score, accuracy_score,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
)

from src.dataset        import *
from src.simpleCNN      import *
from src.efficientNet   import *
from src.trainer        import ArcFaceLightningModule
from src.constant       import *
from sklearn.model_selection import StratifiedKFold
from torchvision        import transforms

In [4]:
import struct
import numpy as np
from collections import Counter
import src.constant as C


idx = {}
with open('dataset/casia-webface/train.idx', 'r') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        key, offset = line.split('\t')
        idx[int(key)] = int(offset)

labels = []
with open('dataset/casia-webface/train.rec', 'rb') as f:
    for offset in idx.values():
        f.seek(offset)
        f.read(C._IR_BUFFER)
        label = struct.unpack(C._IR_FORMAT, f.read(C._IR_SIZE))[1]
        labels.append(int(label))

counts      = Counter(labels)
sorted_counts = sorted(counts.values(), reverse=True)

low_thresh  = int(np.percentile(sorted_counts, 10))
high_thresh = int(np.percentile(sorted_counts, 80))

high_x = next(i for i, c in enumerate(sorted_counts) if c <= high_thresh)
low_x  = next(i for i, c in enumerate(sorted_counts) if c <= low_thresh)

train_ids = {pid for pid, cnt in counts.items() if low_thresh <= cnt <= high_thresh}
test_ids = {pid for pid, cnt in counts.items() if cnt <= low_thresh or cnt >= high_thresh}

print(f"Keep identities with photos between {high_thresh} and {low_thresh}")
print(f"Identities kept  : {sum(low_thresh <= c <= high_thresh for c in counts.values()):,}  [{sum(c if low_thresh <= c <= high_thresh else 0 for c in counts.values())}]")
print(f"Identities total : {len(counts):,} [{sum(c for c in counts.values())}]")

Keep identities with photos between 57 and 15
Identities kept  : 7,824  [214927]
Identities total : 10,572 [501196]


In [ ]:
from torchvision import transforms

# config
IMAGE_SIZE = 112
N_FOLDS = 3
MAX_EPOCH = 30
EARLY_STOPPING_PATIENCE = 5
BATCH_SIZE = 256
PRECISION = 'bf16-mixed'
EMBED_DIM = 256

# testing dataset
TEST_NAMES = ['rec_test', 'lfw', 'cfp_fp', 'agedb_30', 'sllfw', 'talfw']
EVAL_BINS = {
    'lfw'      : 'dataset/eval/lfw.bin',
    'cfp_fp'   : 'dataset/eval/cfp_fp.bin',
    'agedb_30' : 'dataset/eval/agedb_30.bin',
    'sllfw'    : 'dataset/eval/sllfw.bin',
    'talfw'    : 'dataset/eval/talfw.bin',
}

In [34]:
def get_val_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])


def collect_probs(model, dataset, batch_size=64, num_workers=4, device='cuda'):
    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=False, num_workers=num_workers,
                        pin_memory=True)
    all_probs, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for img_a, img_b, label in loader:
            img_a, img_b = img_a.to(device), img_b.to(device)
            emb_a = F.normalize(model.backbone(img_a), p=2, dim=1).float()
            emb_b = F.normalize(model.backbone(img_b), p=2, dim=1).float()
            dist  = F.pairwise_distance(emb_a, emb_b)
            prob  = (1 / (1 + dist)).cpu().numpy()
            all_probs.append(prob)
            all_labels.append(label.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def find_optimal_threshold(probs, labels):
    fpr, tpr, thresholds = roc_curve(labels, probs)
    # top left
    idx = np.argmin((1 - tpr)**2 + fpr**2)
    return float(thresholds[idx]), fpr, tpr, thresholds


def compute_metrics(probs, labels, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        'threshold' : threshold,
        'f1'        : f1_score(labels, preds, average='macro'),
        'acc'       : accuracy_score(labels, preds),
    }


def save_plots(probs, labels, preds, name, log_dir):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    RocCurveDisplay.from_predictions(labels, probs, ax=axes[0], name=name)
    axes[0].set_title(f'{name} — ROC')

    PrecisionRecallDisplay.from_predictions(labels, probs, ax=axes[1], name=name)
    axes[1].set_title(f'{name} — PR curve')

    ConfusionMatrixDisplay.from_predictions(
        labels, preds, display_labels=['diff', 'same'], ax=axes[2]
    )
    axes[2].set_title(f'{name} — confusion matrix')

    plt.suptitle(f'{name} — calibrated', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(log_dir, f'{name}_calibrated.png'), bbox_inches='tight')
    plt.close(fig)

def get_val_dataset(rec_path, idx_path, train_ids, fold, n_folds, seed, val_tf):
    """Reconstruct the exact val split for a given fold."""
    full_ds    = RecDataset(rec_path, idx_path, allowed_ids=train_ids, transform=None)
    all_labels = [full_ds.samples[i][1] for i in range(len(full_ds))]
    skf        = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    splits     = list(skf.split(range(len(full_ds)), all_labels))
    _, val_idx = splits[fold]
    return SiamesePairDataset(
        TransformSubset(full_ds, val_idx, val_tf),
        seed=seed
    )

In [35]:
def main(base_log_dir, rec_path, idx_path, train_ids, backbone,
         n_folds=N_FOLDS, batch_size=BATCH_SIZE, num_workers=4):

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    val_tf = get_val_transform()
    test_names = list(EVAL_BINS.keys())

    all_fold_summaries = []

    for fold in range(n_folds):
        print(f"\n{'═'*55}")
        print(f"  Fold {fold} — calibrating threshold on val split")
        print(f"{'═'*55}")

        fold_dir  = os.path.join(base_log_dir, f'fold_{fold}')
        ckpt_path = os.path.join(fold_dir, 'checkpoints', f'fold{fold}_best.ckpt')
        os.makedirs(fold_dir, exist_ok=True)

        # load model
        model = ArcFaceLightningModule.load_from_checkpoint(
            ckpt_path,
            backbone     = backbone(embed_dim=256),
            map_location = device,
        ).to(device)

        # ── Step 1: calibrate on val fold ──────────────────────────
        print(f"\n[1/2] Running inference on val fold {fold}...")
        val_ds              = get_val_dataset(rec_path, idx_path, train_ids,
                                              fold, n_folds, SEED, val_tf)
        val_probs, val_lbls = collect_probs(model, val_ds, batch_size, num_workers, device)
        threshold, fpr, tpr, thresholds           = find_optimal_threshold(val_probs, val_lbls)

        val_metrics = compute_metrics(val_probs, val_lbls, threshold)
        print(f"  Calibrated threshold : {threshold:.4f}")
        print(f"  Val F1  (calibrated) : {val_metrics['f1']:.4f}")
        print(f"  Val ACC (calibrated) : {val_metrics['acc']:.4f}")

        # save val calibration info
        pd.DataFrame([{'fold': fold, **val_metrics}]).to_csv(
            os.path.join(fold_dir, 'val_calibration.csv'), index=False
        )

        # ── Step 2: apply threshold to test benchmarks ─────────────
        print(f"\n[2/2] Applying threshold={threshold:.4f} to test benchmarks...")
        fold_rows = []

        for name, bin_path in EVAL_BINS.items():
            test_ds               = BinDataset(bin_path, transform=val_tf)
            test_probs, test_lbls = collect_probs(model, test_ds,
                                                  batch_size, num_workers, device)
            metrics               = compute_metrics(test_probs, test_lbls, threshold)
            preds                 = (test_probs >= threshold).astype(int)

            print(f"  {name:<12}  F1={metrics['f1']:.4f}  ACC={metrics['acc']:.4f}")

            # save CSV
            pd.DataFrame({
                'prob'         : test_probs,
                'label'        : test_lbls,
                'pred'         : preds,
                'threshold'    : threshold,
            }).to_csv(os.path.join(fold_dir, f'{name}_calibrated_predictions.csv'), index=False)

            # save plots
            save_plots(test_probs, test_lbls, preds, name, fold_dir)

            fold_rows.append({'fold': fold, 'benchmark': name, **metrics})

        all_fold_summaries.extend(fold_rows)

    # ── Cross-fold summary ──────────────────────────────────────────
    summary_df = pd.DataFrame(all_fold_summaries)
    summary_df.to_csv(os.path.join(base_log_dir, 'calibration_summary.csv'), index=False)

    # mean ± std across folds per benchmark
    print(f"\n\n{'═'*55}")
    print("  Cross-fold summary (mean ± std)")
    print(f"{'═'*55}")
    agg = summary_df.groupby('benchmark')[['f1', 'acc', 'threshold']].agg(
        ['mean', 'std']
    )
    print(agg.to_string())
    agg.to_csv(os.path.join(base_log_dir, 'calibration_summary_agg.csv'))

In [36]:
main(
    base_log_dir='tb_logs/simple_cnn_arcface_face',
    rec_path='dataset/casia-webface/train.rec',
    idx_path='dataset/casia-webface/train.idx',
    train_ids=train_ids,
    backbone=SimpleClassifier
)


═══════════════════════════════════════════════════════
  Fold 0 — calibrating threshold on val split
═══════════════════════════════════════════════════════

[1/2] Running inference on val fold 0...
  Calibrated threshold : 0.4531
  Val F1  (calibrated) : 0.8446
  Val ACC (calibrated) : 0.8446

[2/2] Applying threshold=0.4531 to test benchmarks...
  lfw           F1=0.8685  ACC=0.8698
  cfp_fp        F1=0.7775  ACC=0.7776
  agedb_30      F1=0.6936  ACC=0.6943
  sllfw         F1=0.4232  ACC=0.5333
  talfw         F1=0.7846  ACC=0.7892

═══════════════════════════════════════════════════════
  Fold 1 — calibrating threshold on val split
═══════════════════════════════════════════════════════

[1/2] Running inference on val fold 1...
  Calibrated threshold : 0.4537
  Val F1  (calibrated) : 0.8458
  Val ACC (calibrated) : 0.8459

[2/2] Applying threshold=0.4537 to test benchmarks...
  lfw           F1=0.8644  ACC=0.8660
  cfp_fp        F1=0.7901  ACC=0.7903
  agedb_30      F1=0.7052  ACC

In [38]:
main(
    base_log_dir='tb_logs/effnet_arcface_face',
    rec_path='dataset/casia-webface/train.rec',
    idx_path='dataset/casia-webface/train.idx',
    train_ids=train_ids,
    backbone=EfficientNetBackbone
)


═══════════════════════════════════════════════════════
  Fold 0 — calibrating threshold on val split
═══════════════════════════════════════════════════════

[1/2] Running inference on val fold 0...
  Calibrated threshold : 0.4295
  Val F1  (calibrated) : 0.8802
  Val ACC (calibrated) : 0.8803

[2/2] Applying threshold=0.4295 to test benchmarks...
  lfw           F1=0.9030  ACC=0.9037
  cfp_fp        F1=0.8304  ACC=0.8307
  agedb_30      F1=0.7497  ACC=0.7500
  sllfw         F1=0.5496  ACC=0.6113
  talfw         F1=0.6862  ACC=0.6910

═══════════════════════════════════════════════════════
  Fold 1 — calibrating threshold on val split
═══════════════════════════════════════════════════════

[1/2] Running inference on val fold 1...
  Calibrated threshold : 0.4294
  Val F1  (calibrated) : 0.8799
  Val ACC (calibrated) : 0.8800

[2/2] Applying threshold=0.4294 to test benchmarks...
  lfw           F1=0.9076  ACC=0.9082
  cfp_fp        F1=0.8269  ACC=0.8271


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x000001B4B308C4C0>
Traceback (most recent call last):
  File "c:\Users\ChristianBudhi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "c:\Users\ChristianBudhi\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py", line 1622, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'


  agedb_30      F1=0.7471  ACC=0.7475
  sllfw         F1=0.5453  ACC=0.6083
  talfw         F1=0.6812  ACC=0.6855

═══════════════════════════════════════════════════════
  Fold 2 — calibrating threshold on val split
═══════════════════════════════════════════════════════

[1/2] Running inference on val fold 2...
  Calibrated threshold : 0.4286
  Val F1  (calibrated) : 0.8779
  Val ACC (calibrated) : 0.8780

[2/2] Applying threshold=0.4286 to test benchmarks...
  lfw           F1=0.9058  ACC=0.9063
  cfp_fp        F1=0.8287  ACC=0.8289
  agedb_30      F1=0.7578  ACC=0.7585
  sllfw         F1=0.5600  ACC=0.6180
  talfw         F1=0.6772  ACC=0.6810


═══════════════════════════════════════════════════════
  Cross-fold summary (mean ± std)
═══════════════════════════════════════════════════════
                 f1                 acc           threshold          
               mean       std      mean       std      mean       std
benchmark                                               